In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpbhddtwsr".


# Vector reduction using global memory 

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int N)
{
    int idx = threadIdx.x + blockDim.x * blockIdx.x;
    for(int stride = 1; stride < blockDim.x; stride *= 2)
    {
        __syncthreads();
        if(idx % (2*stride) == 0 && idx + stride < N)
        {
            input[idx] += input[idx + stride];
        }
    }
    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = input[blockIdx.x * blockDim.x];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 16;
    float h_input[16] = {10,1,8,-1,0,-2,3,4,-2,-3, 2,7,0,11,0,2};

    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));
    
    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    reduce_to_place<<<2, 8>>>(d_input, N);
    cudaDeviceSynchronize();
    cudaMemcpy(h_input, d_input, N*sizeof(float), cudaMemcpyDeviceToHost);
    printf("Reduction result: %f\n", h_input[0]);
    reduce_to_place<<<1, 2>>>(d_input, 2);
    cudaMemcpy(h_input, d_input, N*sizeof(float), cudaMemcpyDeviceToHost);
    printf("Final reduction result: %f\n", h_input[0]);
    cudaFree(d_input);
}

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int n)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    for(int stride = 1; stride < blockDim.x; stride *= 2)
    {
        __syncthreads();
        /*if(idx % (2*stride) == 0 && idx + stride < n)
        {
            input[idx] += input[idx + stride];
        }*/
        //this is efficient as modulas operation is expensive as compared to the above commented condition
        if(idx + stride < n)
        {
            input[idx] += input[idx + stride];
        }
    }

    __syncthreads();
    
    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = input[blockIdx.x * blockDim.x];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 1024*1024;
    float *h_input = (float*)malloc(N*sizeof(float));
    for(int i = 0; i < N; i++)
    {
        h_input[i] = i * 1.0f;
    }
    clock_t start = clock();
    float cpu_result = cpu_reduce(h_input, N);
    clock_t end = clock();
    double cpu_time = (double(end - start)) * 1000.0 / CLOCKS_PER_SEC;
    printf("CPU time: %f milliseconds\n", cpu_time);
    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));

    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    
    cudaEvent_t gpu_start, gpu_end;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_end);
    cudaEventRecord(gpu_start);
    
    int blocksize = 256;
    int gridsize = (N+ blocksize - 1)/blocksize;
    int current_size = N;
    while(gridsize > 1)
    {
        reduce_to_place<<<gridsize, blocksize>>>(d_input, current_size);
        cudaDeviceSynchronize();
        current_size = gridsize;
        gridsize = (gridsize + blocksize - 1) / blocksize;
    }
    
    reduce_to_place<<<1, blocksize>>>(d_input, current_size);
    cudaDeviceSynchronize();
    
    cudaEventRecord(gpu_end);
    cudaEventSynchronize(gpu_end);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, gpu_start, gpu_end);
    
    cudaMemcpy(h_input, d_input, sizeof(float), cudaMemcpyDeviceToHost);
    printf("GPU reduction result: %f\n", h_input[0]);
    printf("GPU time: %f milliseconds\n", gpu_time);
    printf("Speedup: %f x\n", cpu_time / gpu_time);
    free(h_input);
    cudaFree(d_input);
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_end);
}

# Vector Reduction and optimization using shared memory

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int N)
{
    __shared__ float shared_data[blockDim.x];
    int idx = threadIdx.x + (2 * blockDim.x * blockIdx.x);

    if(idx + blockDim.x < N)
    {
        shared_data[threadIdx.x] = input[idx] + input[idx + blockDim.x];
    }
    else if(idx < N)
    {
        shared_data[threadIdx.x] = input[idx];
    }
    else
    {
        shared_data[threadIdx.x] = 0.0f;
    }

    __syncthreads();

    for(int stride = blockDim.x >> 1; stride > 0; stride >>= 1)
    {
        if(idx + stride < N)
        {
            shared_data[threadIdx.x] += shared_data[threadIdx.x + stride];
        }
        __syncthreads();
    }

    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = shared_data[0];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 1024*1024;
    float *h_input = (float*)malloc(N*sizeof(float));
    for(int i = 0; i < N; i++)
    {
        h_input[i] = i * 1.0f;
    }
    clock_t start = clock();
    float cpu_result = cpu_reduce(h_input, N);
    clock_t end = clock();
    double cpu_time = (double(end - start)) * 1000.0 / CLOCKS_PER_SEC;
    printf("CPU time: %f milliseconds\n", cpu_time);
    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));

    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    
    cudaEvent_t gpu_start, gpu_end;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_end);
    cudaEventRecord(gpu_start);
    
    int blocksize = 256;
    int gridsize = (N + (2 * blocksize) - 1) / (2 * blocksize);
    int current_size = N;
    while(gridsize > 1)
    {
        reduce_to_place<<<gridsize, blocksize>>>(d_input, current_size);
        cudaDeviceSynchronize();
        current_size = gridsize;
        gridsize = (gridsize + (2 * blocksize) - 1) / (2 * blocksize);
    }
    
    reduce_to_place<<<1, blocksize>>>(d_input, current_size);
    cudaDeviceSynchronize();
    
    cudaEventRecord(gpu_end);
    cudaEventSynchronize(gpu_end);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, gpu_start, gpu_end);
    
    cudaMemcpy(h_input, d_input, sizeof(float), cudaMemcpyDeviceToHost);
    printf("GPU reduction result: %f\n", h_input[0]);
    printf("GPU time: %f milliseconds\n", gpu_time);
    printf("Speedup: %f x\n", cpu_time / gpu_time);
    free(h_input);
    cudaFree(d_input);
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_end);
}

# Vector Reduction and optimization using shared memory

In [ ]:
%%cuda
#include <stdio.h>

__global__ void reduce_to_place(float *input, int N)
{
    __shared__ float shared_data[256];
    int idx = threadIdx.x + (2 * blockDim.x * blockIdx.x);

    if(idx + blockDim.x < N)
    {
        shared_data[threadIdx.x] = input[idx] + input[idx + blockDim.x];
    }
    else if(idx < N)
    {
        shared_data[threadIdx.x] = input[idx];
    }
    else
    {
        shared_data[threadIdx.x] = 0.0f;
    }

    __syncthreads();

    for(int stride = blockDim.x >> 1; stride > 0; stride >>= 1)
    {
        if(threadIdx.x < stride)
        {
            shared_data[threadIdx.x] += shared_data[threadIdx.x + stride];
        }
        __syncthreads();
    }

    if(threadIdx.x == 0)
    {
        input[blockIdx.x] = shared_data[0];
    }
}

float cpu_reduce(float *input, int n)
{
    float sum = 0.0;
    for(int i = 0; i < n; i++)
    {
        sum += input[i];
    }
    return sum;
}

int main()
{
    int N = 1024*1024;
    float *h_input = (float*)malloc(N*sizeof(float));
    for(int i = 0; i < N; i++)
    {
        h_input[i] = i * 1.0f;
    }
    clock_t start = clock();
    float cpu_result = cpu_reduce(h_input, N);
    clock_t end = clock();
    double cpu_time = (double(end - start)) * 1000.0 / CLOCKS_PER_SEC;
    printf("CPU time: %f milliseconds\n", cpu_time);
    printf("CPU reduction result: %f\n", cpu_reduce(h_input, N));

    float *d_input;
    cudaMalloc(&d_input, N*sizeof(float));
    cudaMemcpy(d_input, h_input, N*sizeof(float), cudaMemcpyHostToDevice);
    
    cudaEvent_t gpu_start, gpu_end;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_end);
    cudaEventRecord(gpu_start);
    
    int blocksize = 256;
    int gridsize = (N + (2 * blocksize) - 1) / (2 * blocksize);
    int current_size = N;
    while(gridsize > 1)
    {
        reduce_to_place<<<gridsize, blocksize>>>(d_input, current_size);
        cudaDeviceSynchronize();
        current_size = gridsize;
        gridsize = (gridsize + (2 * blocksize) - 1) / (2 * blocksize);
    }
    
    reduce_to_place<<<1, blocksize>>>(d_input, current_size);
    cudaDeviceSynchronize();
    
    cudaEventRecord(gpu_end);
    cudaEventSynchronize(gpu_end);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, gpu_start, gpu_end);
    
    cudaMemcpy(h_input, d_input, sizeof(float), cudaMemcpyDeviceToHost);
    printf("GPU reduction result: %f\n", h_input[0]);
    printf("GPU time: %f milliseconds\n", gpu_time);
    printf("Speedup: %f x\n", cpu_time / gpu_time);
    free(h_input);
    cudaFree(d_input);
    cudaEventDestroy(gpu_start);
    cudaEventDestroy(gpu_end);
}

# Vector Reduction using the above + shuffle operation optimization

In [8]:
%%cuda

#include <iostream>
#include <cuda_runtime.h>

__inline__ __device__ float warpReduceSum(float val) {
    for (int offset = warpSize >> 1; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xFFFFFFFF, val, offset);
    }
    return val;
}

__global__ void reduce_with_warp_optimization(float* input, int n) {
    extern __shared__ float shared[];  // Shared memory for inter-warp reduction
    int tid = threadIdx.x;
    int index = 2 * blockIdx.x * blockDim.x + tid;
    float sum = 0.0f;
    // Load two elements from global memory into registers (to minimize shared memory usage)
    // This is just to use double size for each vector (512 elements for a block of 256 threads) (more work per thread optimization)
    //once you finish this, you will have 256 elements to work on in the next step.
    sum = (index < n ? input[index] : 0.0f) + (index + blockDim.x < n ? input[index + blockDim.x] : 0.0f);
    // Perform warp-level reduction
    for (int offset = warpSize >>1; offset > 0; offset >>= 1) {
        sum += __shfl_down_sync(0xFFFFFFFF, sum, offset);
    }
    //Now, you have finished the first step, that means that you calculated one partial sum per warp.
    // we need to store these values in the shared memory
    //and then we need to add all of these partial sums together to get a final sum (one element per block From all warps)
    //The final output here is all partial sums will be in the first warp
    if (tid % warpSize == 0) {
        shared[tid / warpSize] = sum;
    }
    __syncthreads();  // Synchronize threads before inter-warp reduction
    //add all partial sums together (from the first warp)
    if (tid < warpSize) {
        sum = (tid < (blockDim.x / warpSize)) ? shared[tid] : 0.0f;
        for (int offset = warpSize >> 1; offset > 0; offset >>= 1) {
        sum += __shfl_down_sync(0xFFFFFFFF, sum, offset);
    }
    }
    // Store the final reduced result for the block in global memory
    if (tid == 0) {
        input[blockIdx.x] = sum;
    }
}

float cpu_reduce(float* input, int size) {
    float sum = 0.0f;
    for (int i = 0; i < size; ++i) {
        sum += input[i];
    }
    return sum;
}

int main() {
    int n = 1024 * 1024;  // Number of elements
    size_t bytes = n * sizeof(float);

    // Host memory allocation
    float* h_input = new float[n];
    float* d_input;

    // Initialize input array
    for (int i = 0; i < n; i++) {
        h_input[i] = static_cast<float>(i + 1);  // Initialize from 1 to n
    }

    // Device memory allocation
    cudaMalloc(&d_input, bytes);

    // Copy data from host to device
    cudaMemcpy(d_input, h_input, bytes, cudaMemcpyHostToDevice);

    // Launch the reduction kernel multiple times
    int blockSize = 256;  // Number of threads per block
    int gridSize = (n + 2 * blockSize - 1) / (2 * blockSize);  // Number of blocks
    size_t sharedMemSize = (blockSize / 32) * sizeof(float);  // Using 32 instead of warpSize


    clock_t start = clock();
    // Calculate the sum using the CPU function for verification
    float total_sum = cpu_reduce(h_input, n);
    std::cout << "Total sum (CPU): " << total_sum << std::endl;
    clock_t end = clock();
    double cpu_time = (double(end - start)) * 1000.0 / CLOCKS_PER_SEC;
    printf("CPU time: %f milliseconds\n", cpu_time);

    cudaEvent_t gpu_start, gpu_end;
    cudaEventCreate(&gpu_start);
    cudaEventCreate(&gpu_end);
    cudaEventRecord(gpu_start);
    // Perform iterative reduction until we have one block left
    while (gridSize > 1) {
        reduce_with_warp_optimization<<<gridSize, blockSize, sharedMemSize>>>(d_input, n);
        cudaDeviceSynchronize();  // Ensure kernel execution completes

        // Update n to reflect the reduced number of elements
        n = gridSize;
        gridSize = (n + 2 * blockSize - 1) / (2 * blockSize);  // Update gridSize for the next iteration
    }

    // Final reduction when gridSize == 1
    reduce_with_warp_optimization<<<1, blockSize, sharedMemSize>>>(d_input, n);
    cudaDeviceSynchronize();  // Ensure final kernel execution completes

    // Copy the final result back to the host (the sum should be in h_input[0])
    cudaMemcpy(h_input, d_input, sizeof(float), cudaMemcpyDeviceToHost);

    cudaEventRecord(gpu_end);
    cudaEventSynchronize(gpu_end);
    float gpu_time = 0;
    cudaEventElapsedTime(&gpu_time, gpu_start, gpu_end);

    // Print the final result
    std::cout << "Final sum (GPU): " << h_input[0] << std::endl;
    printf("GPU time: %f milliseconds\n", gpu_time);
    printf("Speedup: %f x\n", cpu_time / gpu_time);

    // Free memory
    cudaFree(d_input);
    delete[] h_input;

    return 0;
}

Total sum (CPU): 5.49822e+11
CPU time: 3.251000 milliseconds
Final sum (GPU): 5.49756e+11
GPU time: 0.281888 milliseconds
Speedup: 11.532949 x

